# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiknaxTheGreek/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

This notebook keeps the project architecture fixed before model evaluation: **classification + regression + ranking**.

The five model features were already selected in Assignment 4 using March-only information and are not reopened here:

1. `aggregate_ctr`
2. `median_position`
3. `position_slope_per_day`
4. `position_iqr`
5. `content_age_days`

### Classification — Logistic Regression

The binary target is future decline: `future_impression_change < 0`.

A scaled **Logistic Regression** is the first learned classifier because the target is binary, the model produces an interpretable probability needed by the final ranking, and it provides a deliberately simple comparison against the frozen training-prior baseline. No class weighting or test-driven tuning is used.

Primary metric: **ROC-AUC**.

Frozen Assignment 5 baseline: **ROC-AUC = 0.500**.

### Regression — Random Forest Regressor

The continuous target is signed `future_impression_change`.

A deliberately moderate **Random Forest Regressor** is used because the five March features may relate to future movement nonlinearly and through interactions. The configuration is fixed before held-out evaluation:

- `n_estimators=300`
- `max_depth=6`
- `min_samples_leaf=10`
- `random_state=42`
- `n_jobs=-1`

No hyperparameter search is performed against the held-out clients.

Primary metric: **RMSE**.

Frozen Assignment 5 baseline: **RMSE = 1.4311**.

### Ranking — transparent risk × severity score

The learned ranking combines the two model outputs rather than creating a new manual ranking label:

`predicted_decline_severity = max(0, -predicted_future_change)`

`ranking_score = p_decline × predicted_decline_severity`

A larger score therefore means the page is both more likely to decline and predicted to deteriorate more severely. This is a transparent prioritisation score, not a causal-effect estimate.

Primary metric: **Precision@50**.

Frozen Assignment 5 ranking baseline: **Precision@50 = 0.480**.

The six held-out clients remain sealed for model selection. The feature set, model families, hyperparameters, ranking formula, split and primary metrics are fixed before held-out model performance is inspected.

In [ ]:
# STEP 1 — reconstruct the locked March feature frame and load frozen baselines.
# This cell does NOT inspect held-out model performance or tune any method.

import os
import json
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# Hugging Face token stays private.
HF_TOKEN = os.environ.get("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        pass

if not HF_TOKEN:
    raise RuntimeError(
        "HF_TOKEN is missing. Add a Hugging Face READ token as a Colab Secret named HF_TOKEN."
    )

con = duckdb.connect()
safe_token = HF_TOKEN.replace("'", "''")
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{safe_token}')")

REL = "hf://datasets/FlyRank/internship-warehouse"
MARCH = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-03/*.parquet')"
APRIL = f"read_parquet('{REL}/fact_content_daily_performance/month=2026-04/*.parquet')"
DIM_CONTENT = f"read_parquet('{REL}/dim_content.parquet')"

# Reconstruct the exact Assignment-4/5 modeling population.
# April is used here only for the already-locked >=20-day outcome-observability rule;
# no April outcome value is used for feature choice or model selection.
march_cov = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS march_usable_days
    FROM {MARCH}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

april_cov = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        COUNT(DISTINCT report_date) AS april_usable_days
    FROM {APRIL}
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
""").df()

matched_keys = (
    march_cov[march_cov["march_usable_days"] >= 20]
    .merge(april_cov, on=["client_hash_id", "content_hash_id"], how="inner")
)
matched_keys = matched_keys[
    matched_keys["april_usable_days"] >= 20
][["client_hash_id", "content_hash_id"]].drop_duplicates()
con.register("matched_keys", matched_keys)

march_exposure = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN matched_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

march_exposure["exposure_tier"] = pd.cut(
    march_exposure["march_avg_impressions_per_day"],
    bins=[-float("inf"), 13.42, 58.25, float("inf")],
    labels=["Low", "Medium", "High"],
    include_lowest=True,
)

client_tier_counts = (
    march_exposure
    .groupby(["client_hash_id", "exposure_tier"], observed=False)
    .size()
    .unstack(fill_value=0)
    .reindex(columns=["Low", "Medium", "High"], fill_value=0)
)

eligible_clients = client_tier_counts[
    client_tier_counts.min(axis=1) >= 40
].index.tolist()

balanced_poc = (
    march_exposure[march_exposure["client_hash_id"].isin(eligible_clients)]
    .sort_values(["client_hash_id", "exposure_tier", "content_hash_id"])
    .groupby(
        ["client_hash_id", "exposure_tier"],
        observed=False,
        group_keys=False,
    )
    .head(40)
    .reset_index(drop=True)
)

balanced_keys = balanced_poc[
    ["client_hash_id", "content_hash_id"]
].drop_duplicates()
con.register("balanced_keys", balanced_keys)

# Construct only the five already-locked March-safe model features.
march_features = con.sql(f"""
    WITH daily AS (
        SELECT
            f.client_hash_id,
            f.content_hash_id,
            f.report_date,
            DATE_DIFF(
                'day',
                DATE '2026-03-01',
                f.report_date
            )::DOUBLE AS day_index,
            f.gsc_impressions::DOUBLE AS impressions,
            f.gsc_clicks::DOUBLE AS clicks,
            CASE
                WHEN f.gsc_avg_position >= 1
                THEN f.gsc_avg_position::DOUBLE
                ELSE NULL
            END AS valid_position
        FROM {MARCH} AS f
        INNER JOIN balanced_keys AS k
            USING (client_hash_id, content_hash_id)
        WHERE f.gsc_data_available IS TRUE
    )
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(clicks) / NULLIF(SUM(impressions), 0) AS aggregate_ctr,
        MEDIAN(valid_position) AS median_position,
        REGR_SLOPE(valid_position, day_index)
            FILTER (WHERE valid_position IS NOT NULL)
            AS position_slope_per_day,
        (
            QUANTILE_CONT(valid_position, 0.75)
            - QUANTILE_CONT(valid_position, 0.25)
        ) AS position_iqr
    FROM daily
    GROUP BY client_hash_id, content_hash_id
""").df()

age_feature = con.sql(f"""
    SELECT
        d.client_hash_id,
        d.content_hash_id,
        DATE_DIFF(
            'day',
            d.content_created_date,
            DATE '2026-03-31'
        )::DOUBLE AS content_age_days
    FROM {DIM_CONTENT} AS d
    INNER JOIN balanced_keys AS k
        USING (client_hash_id, content_hash_id)
""").df()

FINAL_FEATURES = [
    "aggregate_ctr",
    "median_position",
    "position_slope_per_day",
    "position_iqr",
    "content_age_days",
]

feature_frame = march_features.merge(
    age_feature,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Load Assignment-5 receipts rather than redefining the benchmark.
output_dir = Path("../outputs")
with open(output_dir / "baseline_split_manifest.json", "r", encoding="utf-8") as fh:
    split_manifest = json.load(fh)

with open(output_dir / "multitask_baseline_benchmark.json", "r", encoding="utf-8") as fh:
    frozen_benchmarks = json.load(fh)

# Locked learned-method specification. These choices precede held-out evaluation.
MODEL_SPEC = {
    "classification": {
        "model": "StandardScaler + LogisticRegression",
        "primary_metric": "ROC-AUC",
    },
    "regression": {
        "model": "RandomForestRegressor",
        "n_estimators": 300,
        "max_depth": 6,
        "min_samples_leaf": 10,
        "random_state": 42,
        "n_jobs": -1,
        "primary_metric": "RMSE",
    },
    "ranking": {
        "formula": "p_decline * max(0, -predicted_future_change)",
        "k": 50,
        "primary_metric": "Precision@50",
    },
}

# Contract assertions: fail loudly if prior state has drifted.
assert len(feature_frame) == 2520
assert feature_frame["client_hash_id"].nunique() == 21
assert feature_frame[FINAL_FEATURES].notna().all().all()
assert np.isfinite(feature_frame[FINAL_FEATURES].to_numpy(dtype=float)).all()
assert split_manifest["random_state"] == 42
assert split_manifest["test_size"] == 0.25
assert split_manifest["train_pages"] == 1800
assert split_manifest["test_pages"] == 720
assert len(split_manifest["client_overlap"]) == 0
assert np.isclose(
    frozen_benchmarks["classification"]["roc_auc"], 0.5
)
assert np.isclose(
    frozen_benchmarks["regression"]["rmse"], 1.4311128557344202
)
assert np.isclose(
    frozen_benchmarks["ranking"]["precision_at_50"], 0.48
)

print("ASSIGNMENT 6 — METHOD CONTRACT")
print("Feature rows:", len(feature_frame))
print("Clients:", feature_frame["client_hash_id"].nunique())
print("Features:", FINAL_FEATURES)
print("Train pages:", split_manifest["train_pages"])
print("Test pages:", split_manifest["test_pages"])
print("Client overlap:", len(split_manifest["client_overlap"]))
print("\nFrozen baselines:")
print(
    "Classification ROC-AUC:",
    frozen_benchmarks["classification"]["roc_auc"],
)
print(
    "Regression RMSE:",
    frozen_benchmarks["regression"]["rmse"],
)
print(
    "Ranking Precision@50:",
    frozen_benchmarks["ranking"]["precision_at_50"],
)
print("\nLocked learned methods:")
for task, spec in MODEL_SPEC.items():
    print(task, "->", spec)

display(feature_frame.head(10))


## 2. Split design

Assignment 6 inherits the **exact frozen Assignment 5 validation split** rather than creating a new one.

The split was originally created with:

`GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)`

grouped by `client_hash_id`.

This gives:

- **15 training clients / 1,800 pages**
- **6 held-out clients / 720 pages**
- **0 clients shared between train and test**

The grouped design is required because pages from the same client can share site-level behaviour. A random page split could therefore make the test set artificially easy by allowing the same client to appear on both sides.

The feature window remains **1–31 March 2026**. The future outcome remains **1–30 April 2026**. Every page must have at least 20 usable GSC days in both months.

The three targets/evaluation roles are unchanged:

- **Classification:** `future_decline = 1` when `future_impression_change < 0`.
- **Regression:** continuous signed `future_impression_change`.
- **Ranking relevance:** the same binary future-decline outcome, evaluated at `K = 50`.

This split intentionally preserves the observed client shift found in Assignment 5 rather than hiding it. Training decline prevalence is substantially higher than held-out prevalence, and the mean future change also shifts between train and test. That makes the benchmark harder but more honest: Assignment 6 is testing whether the learned models generalise to unseen clients.

In [ ]:
# STEP 2 — reconstruct the locked future targets and apply the exact frozen client split.
# No model is fitted in this cell.

# Future outcome for the exact locked 2,520-page population.
con.register(
    "model_keys",
    feature_frame[["client_hash_id", "content_hash_id"]].drop_duplicates()
)

march_target = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS march_usable_days,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS march_avg_impressions_per_day
    FROM {MARCH} AS f
    INNER JOIN model_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

april_target = con.sql(f"""
    SELECT
        f.client_hash_id,
        f.content_hash_id,
        COUNT(DISTINCT f.report_date) AS april_usable_days,
        SUM(f.gsc_impressions)::DOUBLE / COUNT(DISTINCT f.report_date)
            AS april_avg_impressions_per_day
    FROM {APRIL} AS f
    INNER JOIN model_keys AS k
        USING (client_hash_id, content_hash_id)
    WHERE f.gsc_data_available IS TRUE
    GROUP BY f.client_hash_id, f.content_hash_id
""").df()

target_frame = march_target.merge(
    april_target,
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

target_frame["future_impression_change"] = (
    target_frame["april_avg_impressions_per_day"]
    - target_frame["march_avg_impressions_per_day"]
) / target_frame["march_avg_impressions_per_day"]

target_frame["future_decline"] = (
    target_frame["future_impression_change"] < 0
).astype(int)

modeling_frame = feature_frame.merge(
    target_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "march_usable_days",
            "april_usable_days",
            "future_impression_change",
            "future_decline",
        ]
    ],
    on=["client_hash_id", "content_hash_id"],
    how="inner",
)

# Reuse the exact pseudonymized client lists frozen in Assignment 5.
train_clients = set(split_manifest["train_clients"])
test_clients = set(split_manifest["test_clients"])

train_frame = modeling_frame[
    modeling_frame["client_hash_id"].isin(train_clients)
].copy()

test_frame = modeling_frame[
    modeling_frame["client_hash_id"].isin(test_clients)
].copy()

# Contract checks.
assert len(modeling_frame) == 2520
assert modeling_frame["client_hash_id"].nunique() == 21
assert (modeling_frame["march_usable_days"] >= 20).all()
assert (modeling_frame["april_usable_days"] >= 20).all()

assert len(train_frame) == split_manifest["train_pages"] == 1800
assert len(test_frame) == split_manifest["test_pages"] == 720
assert train_frame["client_hash_id"].nunique() == 15
assert test_frame["client_hash_id"].nunique() == 6

observed_train_clients = set(train_frame["client_hash_id"].unique())
observed_test_clients = set(test_frame["client_hash_id"].unique())

assert observed_train_clients == train_clients
assert observed_test_clients == test_clients
assert observed_train_clients.isdisjoint(observed_test_clients)

assert set(train_frame.index).isdisjoint(set(test_frame.index))
assert len(train_frame) + len(test_frame) == len(modeling_frame)

# Feature/target separation checks.
for forbidden in [
    "future_impression_change",
    "future_decline",
    "march_usable_days",
    "april_usable_days",
]:
    assert forbidden not in FINAL_FEATURES

X_train = train_frame[FINAL_FEATURES].copy()
X_test = test_frame[FINAL_FEATURES].copy()

y_cls_train = train_frame["future_decline"].astype(int).copy()
y_cls_test = test_frame["future_decline"].astype(int).copy()

y_reg_train = train_frame["future_impression_change"].astype(float).copy()
y_reg_test = test_frame["future_impression_change"].astype(float).copy()

assert X_train.notna().all().all()
assert X_test.notna().all().all()
assert np.isfinite(X_train.to_numpy(dtype=float)).all()
assert np.isfinite(X_test.to_numpy(dtype=float)).all()

# Reproduce the frozen Assignment-5 split statistics exactly.
assert np.isclose(
    y_cls_train.mean(),
    frozen_benchmarks["classification"]["train_decline_prior"],
)
assert np.isclose(
    y_cls_test.mean(),
    frozen_benchmarks["classification"]["test_decline_prevalence"],
)
assert np.isclose(
    y_reg_train.mean(),
    frozen_benchmarks["regression"]["train_mean_future_change"],
)
assert np.isclose(
    y_reg_test.mean(),
    frozen_benchmarks["regression"]["test_mean_future_change"],
)

split_summary = pd.DataFrame(
    [
        {
            "split": "train",
            "pages": len(train_frame),
            "clients": train_frame["client_hash_id"].nunique(),
            "decline_prevalence": y_cls_train.mean(),
            "mean_future_change": y_reg_train.mean(),
        },
        {
            "split": "test",
            "pages": len(test_frame),
            "clients": test_frame["client_hash_id"].nunique(),
            "decline_prevalence": y_cls_test.mean(),
            "mean_future_change": y_reg_test.mean(),
        },
    ]
)

print("ASSIGNMENT 6 — FROZEN SPLIT CHECK")
print("Population pages:", len(modeling_frame))
print("Population clients:", modeling_frame["client_hash_id"].nunique())
print("Train pages:", len(train_frame))
print("Train clients:", train_frame["client_hash_id"].nunique())
print("Test pages:", len(test_frame))
print("Test clients:", test_frame["client_hash_id"].nunique())
print(
    "Client overlap:",
    len(observed_train_clients.intersection(observed_test_clients)),
)
print("Feature count:", len(FINAL_FEATURES))
print("Classification target:", "future_decline")
print("Regression target:", "future_impression_change")
print("Ranking K:", split_manifest["ranking_k"])
print("\nObserved split shift:")
display(split_summary)

print("\nTrain feature frame:")
display(X_train.head())
print("\nTest feature frame:")
display(X_test.head())


## 3. Train + compare vs my baseline

The learned models are trained only on the **1,800 pages from the 15 frozen training clients**. The six held-out clients are used once for evaluation.

No feature selection, split changes, class weighting, hyperparameter search, threshold search, or ranking-formula tuning is performed after seeing held-out results.

The three comparisons are therefore like-for-like:

| Task | Frozen Assignment 5 baseline | Learned method | Primary metric |
|---|---|---|---|
| Classification | Training-prior probability | Scaled Logistic Regression | ROC-AUC |
| Regression | Training-set mean future change | Random Forest Regressor | RMSE |
| Ranking | Low-CTR-for-position rule + staleness boost | Risk × severity score | Precision@50 |

For classification, probabilities are evaluated with ROC-AUC and a fixed 0.5 threshold is used only for the supporting Precision / Recall / F1 metrics.

For regression, the signed March→April relative impression change is predicted directly.

For ranking, each held-out page receives:

`predicted_decline_severity = max(0, -predicted_future_change)`

`ranking_score = p_decline × predicted_decline_severity`

The final comparison table reports each learned result beside its pre-existing Assignment 5 baseline on the same held-out population.

In [ ]:
# STEP 3 — train the locked models and compare them with the frozen baselines.

from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    mean_squared_error,
    mean_absolute_error,
    median_absolute_error,
    r2_score,
    ndcg_score,
)

# -------------------------
# 3A. Classification model
# -------------------------
classification_model = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("logistic_regression", LogisticRegression()),
    ]
)

classification_model.fit(X_train, y_cls_train)

cls_prob_test = classification_model.predict_proba(X_test)[:, 1]
cls_pred_test = (cls_prob_test >= 0.5).astype(int)

classification_model_metrics = {
    "name": "standard_scaler_plus_logistic_regression",
    "roc_auc": float(roc_auc_score(y_cls_test, cls_prob_test)),
    "precision": float(
        precision_score(y_cls_test, cls_pred_test, zero_division=0)
    ),
    "recall": float(
        recall_score(y_cls_test, cls_pred_test, zero_division=0)
    ),
    "f1": float(
        f1_score(y_cls_test, cls_pred_test, zero_division=0)
    ),
}

# -------------------------
# 3B. Regression model
# -------------------------
regression_model = RandomForestRegressor(
    n_estimators=300,
    max_depth=6,
    min_samples_leaf=10,
    random_state=42,
    n_jobs=-1,
)

regression_model.fit(X_train, y_reg_train)
reg_pred_test = regression_model.predict(X_test)

regression_model_metrics = {
    "name": "random_forest_regressor",
    "rmse": float(
        np.sqrt(mean_squared_error(y_reg_test, reg_pred_test))
    ),
    "mae": float(
        mean_absolute_error(y_reg_test, reg_pred_test)
    ),
    "median_absolute_error": float(
        median_absolute_error(y_reg_test, reg_pred_test)
    ),
    "r2": float(
        r2_score(y_reg_test, reg_pred_test)
    ),
}

# -------------------------
# 3C. Learned ranking
# -------------------------
ranking_frame = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_decline",
        "future_impression_change",
    ]
].copy()

# Preserve row alignment with X_test/test_frame.
assert ranking_frame.index.equals(X_test.index)

ranking_frame["p_decline"] = cls_prob_test
ranking_frame["predicted_future_change"] = reg_pred_test
ranking_frame["predicted_decline_severity"] = np.maximum(
    0.0,
    -ranking_frame["predicted_future_change"],
)
ranking_frame["ranking_score"] = (
    ranking_frame["p_decline"]
    * ranking_frame["predicted_decline_severity"]
)

# Deterministic ID tie-breaking only after the learned score.
ranking_frame = ranking_frame.sort_values(
    [
        "ranking_score",
        "client_hash_id",
        "content_hash_id",
    ],
    ascending=[False, True, True],
).reset_index(drop=True)

K = int(split_manifest["ranking_k"])
topk_model = ranking_frame.head(K)

test_relevant = int(ranking_frame["future_decline"].sum())
topk_relevant = int(topk_model["future_decline"].sum())
test_base_rate = float(ranking_frame["future_decline"].mean())

model_precision_at_50 = float(
    topk_model["future_decline"].mean()
)
model_recall_at_50 = float(
    topk_relevant / test_relevant
) if test_relevant else 0.0
model_lift_at_50 = float(
    model_precision_at_50 / test_base_rate
) if test_base_rate else np.nan

model_ndcg_at_50 = float(
    ndcg_score(
        ranking_frame["future_decline"]
        .astype(int)
        .to_numpy()
        .reshape(1, -1),
        ranking_frame["ranking_score"]
        .to_numpy(dtype=float)
        .reshape(1, -1),
        k=K,
        ignore_ties=False,
    )
)

ranking_model_metrics = {
    "name": "risk_times_predicted_decline_severity",
    "test_pages": int(len(ranking_frame)),
    "test_relevant_pages": test_relevant,
    "precision_at_50": model_precision_at_50,
    "recall_at_50": model_recall_at_50,
    "lift_at_50": model_lift_at_50,
    "ndcg_at_50": model_ndcg_at_50,
}

# -------------------------
# 3D. Same-split baseline comparisons
# -------------------------
classification_baseline = frozen_benchmarks["classification"]
regression_baseline = frozen_benchmarks["regression"]
ranking_baseline = frozen_benchmarks["ranking"]

comparison_table = pd.DataFrame(
    [
        {
            "task": "Classification",
            "method": "Frozen baseline",
            "primary_metric": "ROC-AUC",
            "primary_value": classification_baseline["roc_auc"],
            "secondary_1": classification_baseline["precision"],
            "secondary_2": classification_baseline["recall"],
            "secondary_3": classification_baseline["f1"],
        },
        {
            "task": "Classification",
            "method": "Logistic Regression",
            "primary_metric": "ROC-AUC",
            "primary_value": classification_model_metrics["roc_auc"],
            "secondary_1": classification_model_metrics["precision"],
            "secondary_2": classification_model_metrics["recall"],
            "secondary_3": classification_model_metrics["f1"],
        },
        {
            "task": "Regression",
            "method": "Frozen baseline",
            "primary_metric": "RMSE",
            "primary_value": regression_baseline["rmse"],
            "secondary_1": regression_baseline["mae"],
            "secondary_2": regression_baseline["median_absolute_error"],
            "secondary_3": regression_baseline["r2"],
        },
        {
            "task": "Regression",
            "method": "Random Forest",
            "primary_metric": "RMSE",
            "primary_value": regression_model_metrics["rmse"],
            "secondary_1": regression_model_metrics["mae"],
            "secondary_2": regression_model_metrics["median_absolute_error"],
            "secondary_3": regression_model_metrics["r2"],
        },
        {
            "task": "Ranking",
            "method": "Frozen baseline",
            "primary_metric": "Precision@50",
            "primary_value": ranking_baseline["precision_at_50"],
            "secondary_1": ranking_baseline["recall_at_50"],
            "secondary_2": ranking_baseline["lift_at_50"],
            "secondary_3": ranking_baseline["ndcg_at_50"],
        },
        {
            "task": "Ranking",
            "method": "Risk × severity ranking",
            "primary_metric": "Precision@50",
            "primary_value": ranking_model_metrics["precision_at_50"],
            "secondary_1": ranking_model_metrics["recall_at_50"],
            "secondary_2": ranking_model_metrics["lift_at_50"],
            "secondary_3": ranking_model_metrics["ndcg_at_50"],
        },
    ]
)

# Clear task-specific labels for the secondary metrics.
secondary_metric_labels = {
    "Classification": "Precision / Recall / F1",
    "Regression": "MAE / Median AE / R²",
    "Ranking": "Recall@50 / Lift@50 / NDCG@50",
}
comparison_table["secondary_metrics"] = comparison_table["task"].map(
    secondary_metric_labels
)

# Explicit primary-metric deltas.
# Positive classification/ranking delta = better.
# Positive regression improvement = lower RMSE than baseline.
primary_improvement = pd.DataFrame(
    [
        {
            "task": "Classification",
            "baseline": classification_baseline["roc_auc"],
            "model": classification_model_metrics["roc_auc"],
            "improvement": (
                classification_model_metrics["roc_auc"]
                - classification_baseline["roc_auc"]
            ),
            "direction": "higher_is_better",
        },
        {
            "task": "Regression",
            "baseline": regression_baseline["rmse"],
            "model": regression_model_metrics["rmse"],
            "improvement": (
                regression_baseline["rmse"]
                - regression_model_metrics["rmse"]
            ),
            "direction": "lower_is_better",
        },
        {
            "task": "Ranking",
            "baseline": ranking_baseline["precision_at_50"],
            "model": ranking_model_metrics["precision_at_50"],
            "improvement": (
                ranking_model_metrics["precision_at_50"]
                - ranking_baseline["precision_at_50"]
            ),
            "direction": "higher_is_better",
        },
    ]
)

# Machine-readable receipt for later validation and the capstone.
model_benchmark_receipt = {
    "split": frozen_benchmarks["split"],
    "features": FINAL_FEATURES,
    "classification": {
        "baseline": classification_baseline,
        "model": classification_model_metrics,
    },
    "regression": {
        "baseline": regression_baseline,
        "model": regression_model_metrics,
    },
    "ranking": {
        "baseline": ranking_baseline,
        "model": ranking_model_metrics,
        "formula": MODEL_SPEC["ranking"]["formula"],
        "k": K,
    },
}

model_receipt_path = output_dir / "assignment6_model_benchmark.json"
with open(model_receipt_path, "w", encoding="utf-8") as fh:
    json.dump(model_benchmark_receipt, fh, indent=2)

# Sanity checks on predictions and the comparison population.
assert len(cls_prob_test) == len(test_frame) == 720
assert len(reg_pred_test) == len(test_frame) == 720
assert len(ranking_frame) == 720
assert ranking_frame["client_hash_id"].nunique() == 6
assert ranking_frame["ranking_score"].notna().all()
assert np.isfinite(ranking_frame["ranking_score"].to_numpy()).all()
assert 0.0 <= classification_model_metrics["roc_auc"] <= 1.0
assert 0.0 <= ranking_model_metrics["precision_at_50"] <= 1.0
assert regression_model_metrics["rmse"] >= 0.0

print("MODEL VS BASELINE — SAME HELD-OUT CLIENTS")
display(
    comparison_table[
        [
            "task",
            "method",
            "primary_metric",
            "primary_value",
            "secondary_metrics",
            "secondary_1",
            "secondary_2",
            "secondary_3",
        ]
    ]
)

print("\nPRIMARY-METRIC IMPROVEMENT")
display(primary_improvement)

print("\nTOP-50 LEARNED RANKING — FIRST 10")
display(
    ranking_frame[
        [
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "predicted_future_change",
            "predicted_decline_severity",
            "ranking_score",
            "future_decline",
            "future_impression_change",
        ]
    ].head(10)
)

print("\nModel benchmark receipt written:", model_receipt_path)


## 4. Errors and interpretation

Model scores are not enough. This section reads the held-out errors **after** the frozen comparison has been completed.

The goal is diagnostic, not corrective: no model, feature, threshold, split, hyperparameter, or ranking formula is changed after this analysis.

The audit covers four questions:

1. **Classification:** where do false positives and false negatives occur?
2. **Regression:** which held-out pages have the largest absolute prediction errors?
3. **Ranking:** which Top-50 recommendations are wrong, and which real declines were missed?
4. **Feature reliance:** what do the models lean on, and does any feature look suspiciously dominant?

For classification, standardized Logistic Regression coefficients show the direction and relative strength of each feature on the training fit. Held-out permutation importance then checks whether shuffling each feature damages ROC-AUC.

For regression, held-out permutation importance measures the increase in RMSE caused by shuffling each feature.

Three concrete failure examples are displayed for each supervised task where possible. These are examples of model difficulty, not evidence that the features cause the observed future outcomes.

Because the held-out clients differ materially from the training clients, error patterns are also summarized by client. This helps distinguish general model failure from cross-client distribution shift.

In [ ]:
# STEP 4 — error analysis and post-hoc interpretation.
# IMPORTANT: this cell does not tune or refit either model.

from sklearn.inspection import permutation_importance

# -------------------------
# 4A. Classification errors
# -------------------------
classification_errors = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_decline",
        "future_impression_change",
    ]
].copy()

# Preserve the same held-out row order used for prediction.
assert classification_errors.index.equals(X_test.index)

classification_errors["p_decline"] = cls_prob_test
classification_errors["predicted_class"] = cls_pred_test
classification_errors["error_type"] = np.select(
    [
        (
            (classification_errors["future_decline"] == 1)
            & (classification_errors["predicted_class"] == 0)
        ),
        (
            (classification_errors["future_decline"] == 0)
            & (classification_errors["predicted_class"] == 1)
        ),
    ],
    ["false_negative", "false_positive"],
    default="correct",
)

classification_error_summary = (
    classification_errors["error_type"]
    .value_counts()
    .rename_axis("error_type")
    .reset_index(name="pages")
)
classification_error_summary["pct_of_test"] = (
    100.0 * classification_error_summary["pages"]
    / len(classification_errors)
)

classification_by_client = (
    classification_errors
    .groupby("client_hash_id")
    .agg(
        pages=("content_hash_id", "size"),
        actual_decline_rate=("future_decline", "mean"),
        mean_predicted_probability=("p_decline", "mean"),
        false_negatives=(
            "error_type",
            lambda s: int((s == "false_negative").sum()),
        ),
        false_positives=(
            "error_type",
            lambda s: int((s == "false_positive").sum()),
        ),
    )
    .reset_index()
)

# Most confident mistakes are useful concrete cases.
false_negative_examples = (
    classification_errors[
        classification_errors["error_type"] == "false_negative"
    ]
    .sort_values(
        ["p_decline", "client_hash_id", "content_hash_id"],
        ascending=[True, True, True],
    )
    .head(3)
)

false_positive_examples = (
    classification_errors[
        classification_errors["error_type"] == "false_positive"
    ]
    .sort_values(
        ["p_decline", "client_hash_id", "content_hash_id"],
        ascending=[False, True, True],
    )
    .head(3)
)

# Standardized logistic coefficients: direction on the fitted training model.
logistic = classification_model.named_steps["logistic_regression"]
classification_coefficients = pd.DataFrame(
    {
        "feature": FINAL_FEATURES,
        "standardized_coefficient": logistic.coef_[0],
    }
)
classification_coefficients["abs_coefficient"] = (
    classification_coefficients["standardized_coefficient"].abs()
)
classification_coefficients = classification_coefficients.sort_values(
    "abs_coefficient",
    ascending=False,
).reset_index(drop=True)

# Held-out permutation importance for ROC-AUC.
classification_perm = permutation_importance(
    classification_model,
    X_test,
    y_cls_test,
    scoring="roc_auc",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

classification_permutation_importance = pd.DataFrame(
    {
        "feature": FINAL_FEATURES,
        "mean_auc_drop": classification_perm.importances_mean,
        "sd_auc_drop": classification_perm.importances_std,
    }
).sort_values(
    "mean_auc_drop",
    ascending=False,
).reset_index(drop=True)

# -------------------------
# 4B. Regression errors
# -------------------------
regression_errors = test_frame[
    [
        "client_hash_id",
        "content_hash_id",
        "future_impression_change",
    ]
].copy()

assert regression_errors.index.equals(X_test.index)

regression_errors["predicted_future_change"] = reg_pred_test
regression_errors["residual"] = (
    regression_errors["future_impression_change"]
    - regression_errors["predicted_future_change"]
)
regression_errors["absolute_error"] = regression_errors["residual"].abs()

largest_regression_errors = (
    regression_errors
    .sort_values(
        ["absolute_error", "client_hash_id", "content_hash_id"],
        ascending=[False, True, True],
    )
    .head(10)
)

regression_by_client = (
    regression_errors
    .groupby("client_hash_id")
    .agg(
        pages=("content_hash_id", "size"),
        actual_mean_change=("future_impression_change", "mean"),
        predicted_mean_change=("predicted_future_change", "mean"),
        mae=("absolute_error", "mean"),
        rmse=(
            "residual",
            lambda s: float(np.sqrt(np.mean(np.square(s)))),
        ),
    )
    .reset_index()
    .sort_values("rmse", ascending=False)
)

# Held-out permutation importance using negative RMSE.
regression_perm = permutation_importance(
    regression_model,
    X_test,
    y_reg_test,
    scoring="neg_root_mean_squared_error",
    n_repeats=20,
    random_state=42,
    n_jobs=-1,
)

# For neg-RMSE scoring, sklearn reports baseline_score - shuffled_score.
# Positive values therefore mean shuffling the feature worsens RMSE.
regression_permutation_importance = pd.DataFrame(
    {
        "feature": FINAL_FEATURES,
        "mean_rmse_increase": regression_perm.importances_mean,
        "sd_rmse_increase": regression_perm.importances_std,
    }
).sort_values(
    "mean_rmse_increase",
    ascending=False,
).reset_index(drop=True)

# -------------------------
# 4C. Ranking errors
# -------------------------
ranking_audit = ranking_frame.copy()
ranking_audit["model_rank"] = np.arange(1, len(ranking_audit) + 1)
ranking_audit["in_top50"] = ranking_audit["model_rank"] <= K

# Wrong recommendations: top-50 pages that did not decline.
ranking_false_picks = (
    ranking_audit[
        ranking_audit["in_top50"]
        & (ranking_audit["future_decline"] == 0)
    ]
    .sort_values(
        ["model_rank", "client_hash_id", "content_hash_id"]
    )
)

# Important missed declines: true declines outside top 50,
# ordered by most negative realized future change first.
ranking_missed_declines = (
    ranking_audit[
        (~ranking_audit["in_top50"])
        & (ranking_audit["future_decline"] == 1)
    ]
    .sort_values(
        [
            "future_impression_change",
            "ranking_score",
            "client_hash_id",
            "content_hash_id",
        ],
        ascending=[True, False, True, True],
    )
)

ranking_error_summary = pd.DataFrame(
    [
        {
            "error_type": "top50_false_pick",
            "pages": int(len(ranking_false_picks)),
        },
        {
            "error_type": "decline_missed_outside_top50",
            "pages": int(len(ranking_missed_declines)),
        },
    ]
)

ranking_by_client = (
    ranking_audit
    .groupby("client_hash_id")
    .agg(
        pages=("content_hash_id", "size"),
        actual_declines=("future_decline", "sum"),
        top50_selected=("in_top50", "sum"),
        top50_true_declines=(
            "future_decline",
            lambda s: int(
                s[
                    ranking_audit.loc[s.index, "in_top50"]
                ].sum()
            ),
        ),
        mean_ranking_score=("ranking_score", "mean"),
    )
    .reset_index()
)

# -------------------------
# 4D. Compact interpretation receipt
# -------------------------
error_audit_receipt = {
    "classification": {
        "false_negatives": int(
            (classification_errors["error_type"] == "false_negative").sum()
        ),
        "false_positives": int(
            (classification_errors["error_type"] == "false_positive").sum()
        ),
        "top_standardized_coefficients": (
            classification_coefficients.head(3)[
                ["feature", "standardized_coefficient"]
            ].to_dict(orient="records")
        ),
        "top_permutation_features": (
            classification_permutation_importance.head(3)[
                ["feature", "mean_auc_drop", "sd_auc_drop"]
            ].to_dict(orient="records")
        ),
    },
    "regression": {
        "largest_absolute_error": float(
            regression_errors["absolute_error"].max()
        ),
        "median_absolute_error_observed": float(
            regression_errors["absolute_error"].median()
        ),
        "top_permutation_features": (
            regression_permutation_importance.head(3)[
                ["feature", "mean_rmse_increase", "sd_rmse_increase"]
            ].to_dict(orient="records")
        ),
    },
    "ranking": {
        "top50_false_picks": int(len(ranking_false_picks)),
        "declines_missed_outside_top50": int(len(ranking_missed_declines)),
        "top50_true_declines": int(
            topk_model["future_decline"].sum()
        ),
    },
}

error_receipt_path = output_dir / "assignment6_error_audit.json"
with open(error_receipt_path, "w", encoding="utf-8") as fh:
    json.dump(error_audit_receipt, fh, indent=2)

# -------------------------
# 4E. Sanity checks
# -------------------------
assert (
    classification_error_summary["pages"].sum()
    == len(test_frame)
    == 720
)
assert len(regression_errors) == 720
assert len(ranking_audit) == 720
assert (
    len(ranking_false_picks)
    + int(topk_model["future_decline"].sum())
    == K
)
assert set(classification_coefficients["feature"]) == set(FINAL_FEATURES)
assert set(
    classification_permutation_importance["feature"]
) == set(FINAL_FEATURES)
assert set(
    regression_permutation_importance["feature"]
) == set(FINAL_FEATURES)

print("CLASSIFICATION ERROR SUMMARY")
display(classification_error_summary)

print("\nCLASSIFICATION ERRORS BY HELD-OUT CLIENT")
display(classification_by_client)

print("\nTHREE MOST CONFIDENT FALSE NEGATIVES")
display(false_negative_examples)

print("\nTHREE MOST CONFIDENT FALSE POSITIVES")
display(false_positive_examples)

print("\nLOGISTIC REGRESSION — STANDARDIZED COEFFICIENTS")
display(classification_coefficients)

print("\nCLASSIFICATION PERMUTATION IMPORTANCE — HELD-OUT ROC-AUC")
display(classification_permutation_importance)

print("\nLARGEST REGRESSION ERRORS")
display(largest_regression_errors)

print("\nREGRESSION ERRORS BY HELD-OUT CLIENT")
display(regression_by_client)

print("\nREGRESSION PERMUTATION IMPORTANCE — HELD-OUT RMSE")
display(regression_permutation_importance)

print("\nRANKING ERROR SUMMARY")
display(ranking_error_summary)

print("\nTHREE TOP-50 FALSE PICKS")
display(
    ranking_false_picks[
        [
            "model_rank",
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "predicted_future_change",
            "ranking_score",
            "future_impression_change",
        ]
    ].head(3)
)

print("\nTHREE LARGE REAL DECLINES MISSED OUTSIDE TOP 50")
display(
    ranking_missed_declines[
        [
            "model_rank",
            "client_hash_id",
            "content_hash_id",
            "p_decline",
            "predicted_future_change",
            "ranking_score",
            "future_impression_change",
        ]
    ].head(3)
)

print("\nRANKING BY HELD-OUT CLIENT")
display(ranking_by_client)

print("\nError-audit receipt written:", error_receipt_path)


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.